# Solar Installation Timeline in California

This notebook calculates the median time from solar interconnection application to utility approval (Permission to Operate) using public CPUC data downloadable from [DGStats](https://www.californiadgstats.ca.gov/downloads/). At the time of download, the latest accessible data from DGStats was from May 31, 2026.

This analysis is done by [SolarWAVE Action](https://solarwaveaction.org/), a California based nonprofit promoting clean energy. All our code is open source and can be found in our [GitHub](https://github.com/SolarWAVE-Action/solarwave-analysis) repository.

Below, Python packages are imported and the application data from Pacific Gas & Electric (PG&E), Southern California Edison (SCE) and San Diego Gas & Electric (SDG&E) from the last 5 years are consolidated. 

In [1]:
import datetime
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import data_analysis as data_analysis
import data_visualization as data_vis

import plotly.io as pio
pio.templates.default = 'plotly_white'
pio.renderers.default = 'notebook'

In [ ]:
data_dir = '/Users/jenny.folkesson/Data/solar/'
dgstats_dir = os.path.join(data_dir, 'applications_20260531/')

df_total = data_analysis.read_stored_data(dgstats_dir, "dgstats_preprocessed_data.csv")

Reading existing file:  /Users/jenny.folkesson/Data/solar/applications_20260531/dgstats_preprocessed_data.csv


DGStats have 3 time related metrics: 
- 'App Recieved Date': The date the application was received by the utility
- 'App Complete Date': The date the application was deemed complete by the utility
- 'App Approved Date': The date the application was approved and a Permission to Operate (PTO) was issued to the customer

The clearest picture of how long a solar application takes in California is obtained by finding the number of days between App Recieved Date and App Approved Date.

In [ ]:
df_total['App Days'] = ((df_total['App Approved Date'] - df_total['App Received Date']).dt.days)

Next, the median number of application days ('App Days') are calculated on an annual basis.

In [ ]:
df = df_total[['App Approved Date', 'System Size DC', 'App Days']]
df = df.set_index('App Approved Date').rename_axis(None)
df = df.resample("YE").agg({'System Size DC': 'sum', 'App Days': 'median'})
df['Year'] = df.index
df['Year'] = df['Year'].dt.year
df = df[df['Year'] > 2016]

Below, the median solar installation times in days for all customer sectors (including residential, commercial, industrial, non-profit, educational, military and other government) is displayed for each year for the last five years (2026 is incomplete).

In [ ]:
df[['Year', 'App Days']]

Below, the same information is visualized in a bar graph.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df['Year'],
    y=df['App Days'],
    hovertext=df['App Days'],
    hovertemplate='%{hovertext}<br>Median application time: %{y:.1f} days',
))
fig.update_layout(
    margin=dict(l=40, r=20, t=20, b=20),
    autosize=True,
    yaxis_title='Median Application Time (days)',
    xaxis_title='Year',
)
fig.update_xaxes(
    dtick="M12"
)
fig.show()

The median time for a solar installation from the day the application was received to complete is 29 days in 2026 so far.

This can be compared to numbers from the National Laboratory of the Rockies (NLR), which was formerly known as National Renewable Energy Laboratory (NREL) but was renamed by the Trump administation. Their [SolarTRACE Data Viewer](https://maps.nlr.gov/solarTRACE/data-viewer) shows a median timeline for all California solar project sizes in **2025 of 26 days** (found by adding the median numbers for CA Permitting, CA Pre-Interconnection, CA - Inspection, and CA - Post Interconnection).
That is the exact same median time that this analysis found for 2025; 26 days.

**SolarWAVE Action found a median installation time of 29 days in 2026 for California solar projects.**
Our analysis method is shared openly for inspection and is consistent with findings by NLR. For any questions or comments, please reach out to hello@solarwaveaction.org.